# Colab实验Notebook

这是当前论文流程的 Colab 专用版本。

本版本只保留你当前实际使用的实验路径：

- 项目目录固定为 `/content/drive/MyDrive/project`
- 真实图像由你手动放在 `data/real_samples/`
- 生成器固定为 `fairdiffusion`
- 检测器固定为 `CNNDetection + LGrad + NPR`
- 保留从生成到主报告的完整流程

说明：

- 所有脚本调用统一使用 `python -u`，便于 VS Code 连接 Colab 时实时查看输出
- 所有运行路径统一走 `PROJECT_ROOT`
- 所有推理脚本统一走 `DEVICE` 变量


## 1. 挂载 Google Drive

作用：

- 挂载你的 Google Drive
- 访问 `/content/drive/MyDrive/project` 项目目录
- 后续所有数据和结果都持久化保存在 Drive 中


In [ ]:
print("hello")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 中央配置

作用：

- 集中管理项目路径、模型、样本数、设备等参数
- 后续所有命令统一引用这里的配置


In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/project"
MODEL_ID = "runwayml/stable-diffusion-v1-5"
DEVICE = "cuda"
SAMPLES_PER_GROUP = 100
REAL_PER_GROUP = 100
SEED = 42
WIDTH = 768
HEIGHT = 768
STEPS = 35
GENERATE_BATCH_SIZE = 4
TORCH_DTYPE = "float16"
ENABLE_XFORMERS = False
ENABLE_VAE_SLICING = True
ENABLE_VAE_TILING = False
CNN_BATCH_SIZE = 48
CNN_NUM_WORKERS = 4
CNN_PIN_MEMORY = True
CNN_AMP = True

NPR_BATCH_SIZE = 32
NPR_NUM_WORKERS = 4
NPR_PIN_MEMORY = True
NPR_AMP = True

LGRAD_BATCH_SIZE = 8
LGRAD_NUM_WORKERS = 2
LGRAD_PIN_MEMORY = True
LGRAD_AMP = True

USE_CLIP = True
CLIP_BATCH_SIZE = 32
CLIP_TEXT_MODE = "profession"
STRICT_GROUP_CHECK = False
CLIP_MIN_SCORE = 0.16
GROUP_MARGIN_MIN = 0.00
HUMAN_PHOTO_MIN = 0.00
DISABLE_CARTOON_CHECK = True
DISABLE_TOY_CHECK = False
ALIGN_ON = "clip"

BOOTSTRAP_ITERS = 1000
DETECTORS = "cnndetection,lgrad,npr"

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"DEVICE = {DEVICE}")
print(f"DETECTORS = {DETECTORS}")
print(f"GEN: {WIDTH}x{HEIGHT}, steps={STEPS}, batch={GENERATE_BATCH_SIZE}, dtype={TORCH_DTYPE}")
print(f"CLIP: batch={CLIP_BATCH_SIZE}, use_clip={USE_CLIP}")
print(f"CNN: batch={CNN_BATCH_SIZE}, workers={CNN_NUM_WORKERS}, amp={CNN_AMP}")
print(f"NPR: batch={NPR_BATCH_SIZE}, workers={NPR_NUM_WORKERS}, amp={NPR_AMP}")
print(f"LGRAD: batch={LGRAD_BATCH_SIZE}, workers={LGRAD_NUM_WORKERS}, amp={LGRAD_AMP}")


## 3. 进入项目目录并检查基础结构

作用：

- 进入项目目录
- 检查关键脚本和依赖文件是否存在
- 避免后面运行到一半才发现路径错误


In [ ]:
%cd /content/drive/MyDrive/project

from pathlib import Path

project_root = Path(PROJECT_ROOT)
assert project_root.exists(), f"项目根目录不存在: {PROJECT_ROOT}"

required_paths = [
    project_root / "requirements.txt",
    project_root / "scripts" / "01_generate.py",
    project_root / "scripts" / "02_quality_filter.py",
    project_root / "scripts" / "03_run_detectors.py",
    project_root / "scripts" / "04_fairness_eval.py",
    project_root / "scripts" / "05_gradcam_analysis.py",
    project_root / "scripts" / "06_patch_shuffling_exp.py",
    project_root / "scripts" / "06_consolidate_results.py",
    project_root / "scripts" / "07_physical_consistency.py",
    project_root / "scripts" / "08_high_pass_innovation.py",
    project_root / "scripts" / "09_master_report.py",
    project_root / "semantic-image-editing-main" / "semantic-image-editing-main",
]

missing = [str(p) for p in required_paths if not p.exists()]
assert not missing, "缺失以下关键路径:\n" + "\n".join(missing)
print("项目结构检查通过")


## 4. 安装项目依赖

作用：

- 安装 `requirements.txt` 中的依赖
- 优先使用 `diffusers` 官方语义编辑管线，不强制依赖本地 `semdiffusers`
- 为后续生成、检测和分析步骤准备环境


In [ ]:
!pip install -U pip
!pip install -r {PROJECT_ROOT}/requirements.txt


## 5. 验证依赖导入

作用：

- 验证关键 Python 包可以正常导入
- 提前发现依赖安装失败的问题


In [ ]:
import importlib
for pkg in ["torch", "transformers", "diffusers", "accelerate"]:
    importlib.import_module(pkg)

from diffusers import SemanticStableDiffusionPipeline
print("关键依赖导入通过")
print("SemanticStableDiffusionPipeline import ok")


## 6. 检查 GPU

作用：

- 确认 Colab 是否真的分配到了 GPU
- 如果没有 GPU，则自动回退到 CPU，避免后续脚本直接报错


In [ ]:
import torch

if DEVICE == "cuda" and torch.cuda.is_available():
    print(f"GPU 可用: {torch.cuda.get_device_name(0)}")
    print(f"GPU 显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
elif DEVICE == "cuda":
    print("未检测到 GPU，DEVICE 自动改为 cpu")
    DEVICE = "cpu"
else:
    print("当前使用 CPU 模式")

print(f"最终 DEVICE = {DEVICE}")


## 7. 检查真实图像目录

作用：

- 确认你手动准备的真实图像已经按四个群组放好
- 本 notebook 不使用任何爬虫

要求目录：

- `data/real_samples/male-doctor`
- `data/real_samples/female-doctor`
- `data/real_samples/male-nurse`
- `data/real_samples/female-nurse`


In [ ]:
real_root = project_root / "data" / "real_samples"
groups = ["male-doctor", "female-doctor", "male-nurse", "female-nurse"]
image_exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

assert real_root.exists(), f"真实图像根目录不存在: {real_root}"

for group in groups:
    group_dir = real_root / group
    assert group_dir.exists(), f"缺失真实图像目录: {group_dir}"
    count = sum(1 for p in group_dir.iterdir() if p.is_file() and p.suffix.lower() in image_exts)
    print(f"{group}: {count}")


## 8. 检查检测器模型代码与权重

作用：

- 确认当前实验需要的官方模型目录和权重都已经就位
- 当前 detector 集合为 `CNNDetection + LGrad + NPR`


In [ ]:
external_root = project_root / ".external_models"
required_model_paths = [
    external_root / "weights" / "cnndetect" / "blur_jpg_prob0.5.pth",
    external_root / "weights" / "lgrad" / "LGrad-4class-Trainon-Progan_car_cat_chair_horse.pth",
    external_root / "weights" / "npr" / "NPR.pth",
    external_root / "weights" / "preprocessing" / "karras2019stylegan-bedrooms-256x256_discriminator.pth",
    external_root / "CNNDetection",
    external_root / "LGrad",
    external_root / "NPR-DeepfakeDetection",
]

missing_model_paths = [str(p) for p in required_model_paths if not p.exists()]
assert not missing_model_paths, "缺失以下模型文件或目录:\n" + "\n".join(missing_model_paths)

for rel in [
    ".external_models/weights/cnndetect",
    ".external_models/weights/lgrad",
    ".external_models/weights/npr",
    ".external_models/weights/preprocessing",
]:
    print("====", rel, "====")
    !ls -lh $PROJECT_ROOT/$rel


## 9. 阶段 01：生成假图并注册真实图，得到原始元数据

作用：

- 使用 `fairdiffusion` 生成假图
- 将你本地已有的真实图像注册进元数据
- 输出 `data/metadata_raw.csv`

说明：

- 当前 notebook 已按高算力 GPU 场景配置为更高分辨率和批量生成
- 如果你的 20GB+ 显存还有余量，可以继续把 `GENERATE_BATCH_SIZE` 提到 6 或 8


In [ ]:
generate_cmd = f"""
python -u {PROJECT_ROOT}/scripts/01_generate.py \
  --project-root {PROJECT_ROOT} \
  --real-source local \
  --generator fairdiffusion \
  --model-id {MODEL_ID} \
  --samples-per-group {SAMPLES_PER_GROUP} \
  --real-per-group {REAL_PER_GROUP} \
  --width {WIDTH} \
  --height {HEIGHT} \
  --steps {STEPS} \
  --batch-size {GENERATE_BATCH_SIZE} \
  --device {DEVICE} \
  --torch-dtype {TORCH_DTYPE} \
  {'--enable-xformers' if ENABLE_XFORMERS else ''} \
  {'--disable-vae-slicing' if not ENABLE_VAE_SLICING else ''} \
  {'--enable-vae-tiling' if ENABLE_VAE_TILING else ''} \
  --seed {SEED} \
  --overwrite
""".strip()
print(generate_cmd)
!{generate_cmd}


## 10. 检查阶段 01 输出

作用：

- 确认 `metadata_raw.csv` 已生成
- 查看各 group 与 `y_true` 的计数是否合理


In [ ]:
import pandas as pd

metadata_raw_path = project_root / "data" / "metadata_raw.csv"
assert metadata_raw_path.exists(), f"缺失阶段01输出: {metadata_raw_path}"

metadata_raw = pd.read_csv(metadata_raw_path)
assert len(metadata_raw) > 0, "metadata_raw.csv 为空"

print(metadata_raw.head())
print(metadata_raw.groupby(["group", "y_true"]).size())


## 11. 阶段 02：质量过滤与分组平衡

作用：

- 计算质量分数、CLIP 图文分数和语义一致性分数
- 使用以职业为中心的中性 CLIP 文本，减少后处理阶段对性别语义的二次放大
- 对真实图和假图使用同一套保守过滤逻辑
- 在每个 `group x y_true` 内做排序，并最终做组间等量保留
- 输出 `data/metadata_balanced.csv`

说明：

- 当前主实验默认使用 `CLIP_TEXT_MODE="profession"`，不直接复用生成 prompt
- 当前主实验默认不启用严格 `pred_group == group` 硬匹配，以免把 FairDiffusion 生成结果再次筛坏
- 默认会复用 `data/metadata_scored.csv`，如果输入文件路径未变化，就不会重复计算质量分数和 CLIP 分数
- 如果你想强制从头重算，可以在命令末尾额外加 `--no-reuse-scored`


In [ ]:
quality_cmd = f"""
python -u {PROJECT_ROOT}/scripts/02_quality_filter.py \
  --project-root {PROJECT_ROOT} \
  --metadata-in {PROJECT_ROOT}/data/metadata_raw.csv \
  --metadata-out {PROJECT_ROOT}/data/metadata_balanced.csv \
  --balanced-dir {PROJECT_ROOT}/data/generated_balanced \
  {'--use-clip' if USE_CLIP else ''} \
  --clip-text-mode {CLIP_TEXT_MODE} \
  --clip-batch-size {CLIP_BATCH_SIZE} \
  --device {DEVICE} \
  --clip-min-score {CLIP_MIN_SCORE} \
  --group-margin-min {GROUP_MARGIN_MIN} \
  --human-photo-min {HUMAN_PHOTO_MIN} \
  {'--strict-group-check' if STRICT_GROUP_CHECK else ''} \
  {'--disable-cartoon-check' if DISABLE_CARTOON_CHECK else ''} \
  {'--disable-toy-check' if DISABLE_TOY_CHECK else ''} \
  --align-on {ALIGN_ON} \
  --copy-files
""".strip()
quality_cmd = " ".join(quality_cmd.split())
print(quality_cmd)
!{quality_cmd}


## 12. 检查阶段 02 输出

作用：

- 确认 `metadata_balanced.csv` 已生成
- 查看过滤前后摘要
- 查看每个 `group x y_true` 的过滤率与保留率
- 查看平衡后的组别统计


In [ ]:
metadata_balanced_path = project_root / "data" / "metadata_balanced.csv"
assert metadata_balanced_path.exists(), f"缺失阶段02输出: {metadata_balanced_path}"

metadata_balanced = pd.read_csv(metadata_balanced_path)
assert len(metadata_balanced) > 0, "metadata_balanced.csv 为空"

print(metadata_balanced.groupby(["group", "y_true"]).size())
print("总数:", len(metadata_balanced))

!cat {PROJECT_ROOT}/results/fairness_tables/quality_clip_summary_before.csv
!cat {PROJECT_ROOT}/results/fairness_tables/quality_clip_summary_after.csv
!cat {PROJECT_ROOT}/results/fairness_tables/quality_clip_filter_audit.csv


## 13. 阶段 03：运行检测器

作用：

- 对平衡后的数据运行 `CNNDetection + LGrad + NPR`
- 输出 detector score CSV 和 summary 文本

说明：

- 这里只保留一次统一运行，不再重复逐个/批量各跑一遍


In [ ]:
!python -u {PROJECT_ROOT}/scripts/03_run_detectors.py --project-root {PROJECT_ROOT} --metadata-in {PROJECT_ROOT}/data/metadata_balanced.csv --detector cnndetection --device {DEVICE} --batch-size {CNN_BATCH_SIZE} --num-workers {CNN_NUM_WORKERS} {'--pin-memory' if CNN_PIN_MEMORY else ''} {'--amp' if CNN_AMP else ''}
!python -u {PROJECT_ROOT}/scripts/03_run_detectors.py --project-root {PROJECT_ROOT} --metadata-in {PROJECT_ROOT}/data/metadata_balanced.csv --detector npr --device {DEVICE} --batch-size {NPR_BATCH_SIZE} --num-workers {NPR_NUM_WORKERS} {'--pin-memory' if NPR_PIN_MEMORY else ''} {'--amp' if NPR_AMP else ''}
!python -u {PROJECT_ROOT}/scripts/03_run_detectors.py --project-root {PROJECT_ROOT} --metadata-in {PROJECT_ROOT}/data/metadata_balanced.csv --detector lgrad --device {DEVICE} --batch-size {LGRAD_BATCH_SIZE} --num-workers {LGRAD_NUM_WORKERS} {'--pin-memory' if LGRAD_PIN_MEMORY else ''} {'--amp' if LGRAD_AMP else ''}

!ls -lh {PROJECT_ROOT}/results/detector_outputs
!cat {PROJECT_ROOT}/results/detector_outputs/cnndetection_summary.txt
!cat {PROJECT_ROOT}/results/detector_outputs/lgrad_summary.txt
!cat {PROJECT_ROOT}/results/detector_outputs/npr_summary.txt


## 14. 阶段 04：公平性评估

作用：

- 分别对三个检测器做公平性评估
- 输出公平性表格、统计量和 bootstrap 区间


In [ ]:
!python -u {PROJECT_ROOT}/scripts/04_fairness_eval.py --detector-csv {PROJECT_ROOT}/results/detector_outputs/cnndetection_scores.csv --output-dir {PROJECT_ROOT}/results/fairness_tables/cnndetection --split test --bootstrap-iters {BOOTSTRAP_ITERS} --seed {SEED}
!python -u {PROJECT_ROOT}/scripts/04_fairness_eval.py --detector-csv {PROJECT_ROOT}/results/detector_outputs/lgrad_scores.csv --output-dir {PROJECT_ROOT}/results/fairness_tables/lgrad --split test --bootstrap-iters {BOOTSTRAP_ITERS} --seed {SEED}
!python -u {PROJECT_ROOT}/scripts/04_fairness_eval.py --detector-csv {PROJECT_ROOT}/results/detector_outputs/npr_scores.csv --output-dir {PROJECT_ROOT}/results/fairness_tables/npr --split test --bootstrap-iters {BOOTSTRAP_ITERS} --seed {SEED}

!find {PROJECT_ROOT}/results/fairness_tables -maxdepth 2 -type f
!cat {PROJECT_ROOT}/results/fairness_tables/cnndetection/fairness_summary.json
!cat {PROJECT_ROOT}/results/fairness_tables/lgrad/fairness_summary.json
!cat {PROJECT_ROOT}/results/fairness_tables/npr/fairness_summary.json


## 15. 阶段 06：整合最新主要结果

作用：

- 将当前主检测器集合的公平性结果整合到总览表
- 输出 overview CSV 和 notes 文本


In [ ]:
!python -u {PROJECT_ROOT}/scripts/06_consolidate_results.py --project-root {PROJECT_ROOT} --metadata {PROJECT_ROOT}/data/metadata_balanced.csv --detectors {DETECTORS} --output-csv {PROJECT_ROOT}/results/fairness_tables/latest_run_overview.csv --output-notes {PROJECT_ROOT}/results/fairness_tables/latest_run_notes.md

!cat {PROJECT_ROOT}/results/fairness_tables/latest_run_notes.md

latest_overview = pd.read_csv(f'{PROJECT_ROOT}/results/fairness_tables/latest_run_overview.csv')
latest_overview.head()


## 16. 阶段 05：Grad-CAM 归因

作用：

- 为 `CNNDetection + LGrad + NPR` 生成归因图
- 当前源码已经对齐到你的 detector 集合


In [ ]:
!python -u {PROJECT_ROOT}/scripts/05_gradcam_analysis.py --project-root {PROJECT_ROOT} --detector-csv {PROJECT_ROOT}/results/detector_outputs/cnndetection_scores.csv --output-dir {PROJECT_ROOT}/results/attribution/cnndetection --analyze-all --max-per-group 2 --device {DEVICE}
!python -u {PROJECT_ROOT}/scripts/05_gradcam_analysis.py --project-root {PROJECT_ROOT} --detector-csv {PROJECT_ROOT}/results/detector_outputs/lgrad_scores.csv --output-dir {PROJECT_ROOT}/results/attribution/lgrad --analyze-all --max-per-group 2 --device {DEVICE}
!python -u {PROJECT_ROOT}/scripts/05_gradcam_analysis.py --project-root {PROJECT_ROOT} --detector-csv {PROJECT_ROOT}/results/detector_outputs/npr_scores.csv --output-dir {PROJECT_ROOT}/results/attribution/npr --analyze-all --max-per-group 2 --device {DEVICE}

!find {PROJECT_ROOT}/results/attribution -maxdepth 3 -type f


## 17. 阶段 06：补丁混洗结构归因

作用：

- 测试检测器更依赖局部纹理还是全局结构
- 使用你当前 detector 集合统一运行


In [ ]:
!python -u {PROJECT_ROOT}/scripts/06_patch_shuffling_exp.py --project-root {PROJECT_ROOT} --patch-scales 1,2,4,8,16,32 --detectors {DETECTORS} --seed {SEED}
!find {PROJECT_ROOT}/results/structural_attribution -maxdepth 2 -type f


## 18. 阶段 07：物理一致性分析

作用：

- 提取真实图与假图的二阶物理统计特征
- 输出物理一致性分析结果


In [ ]:
!python -u {PROJECT_ROOT}/scripts/07_physical_consistency.py --project-root {PROJECT_ROOT} --max-samples 50
!ls -lh {PROJECT_ROOT}/data/physical_consistency_results.csv


## 19. 阶段 08：生成高通残差

作用：

- 生成真实图与假图的高通残差版本
- 为后续残差域 detector 评估做准备


In [ ]:
!python -u {PROJECT_ROOT}/scripts/08_high_pass_innovation.py --project-root {PROJECT_ROOT} --process-all
!find {PROJECT_ROOT}/data/high_pass_residuals -maxdepth 3 -type d


## 20. 在高通残差上运行当前检测器

作用：

- 在 `high_pass_residuals` 上重新运行 `CNNDetection + LGrad + NPR`
- 输出残差域检测结果


In [ ]:
!python -u {PROJECT_ROOT}/scripts/03_run_detectors.py --project-root {PROJECT_ROOT} --input-dir {PROJECT_ROOT}/data/high_pass_residuals --detector cnndetection --output-dir {PROJECT_ROOT}/results/detector_outputs_highpass --device {DEVICE} --batch-size {CNN_BATCH_SIZE} --num-workers {CNN_NUM_WORKERS} {'--pin-memory' if CNN_PIN_MEMORY else ''} {'--amp' if CNN_AMP else ''}
!python -u {PROJECT_ROOT}/scripts/03_run_detectors.py --project-root {PROJECT_ROOT} --input-dir {PROJECT_ROOT}/data/high_pass_residuals --detector npr --output-dir {PROJECT_ROOT}/results/detector_outputs_highpass --device {DEVICE} --batch-size {NPR_BATCH_SIZE} --num-workers {NPR_NUM_WORKERS} {'--pin-memory' if NPR_PIN_MEMORY else ''} {'--amp' if NPR_AMP else ''}
!python -u {PROJECT_ROOT}/scripts/03_run_detectors.py --project-root {PROJECT_ROOT} --input-dir {PROJECT_ROOT}/data/high_pass_residuals --detector lgrad --output-dir {PROJECT_ROOT}/results/detector_outputs_highpass --device {DEVICE} --batch-size {LGRAD_BATCH_SIZE} --num-workers {LGRAD_NUM_WORKERS} {'--pin-memory' if LGRAD_PIN_MEMORY else ''} {'--amp' if LGRAD_AMP else ''}
!ls -lh {PROJECT_ROOT}/results/detector_outputs_highpass


## 21. 阶段 09：主报告

作用：

- 汇总公平性、Grad-CAM、补丁混洗、物理一致性和高通残差结果
- 生成最终总报告


In [ ]:
!python -u {PROJECT_ROOT}/scripts/09_master_report.py --project-root {PROJECT_ROOT} --detectors {DETECTORS}
!find {PROJECT_ROOT}/results/master_report -maxdepth 2 -type f
!cat {PROJECT_ROOT}/results/master_report/MASTER_REPORT.md


## 最后说明

这个 notebook 现在做了以下统一：

- 只保留 Colab 版本，不再混入本地运行分支
- 删除重复挂载和重复运行 detector 的单元
- 所有路径统一走 `PROJECT_ROOT`
- 所有运行脚本统一用 `python -u`
- 所有 detector / generator / device 配置统一走中央配置
- 流程完整保留：生成 -> 质量过滤 -> 检测 -> 公平性 -> 归因 -> 高通残差 -> 主报告
